# Hello Nextmap!

## How to Setup

`Nextmap` supports a pure Python backend and a C++ backend called `emapcc` which is more efficient for large designs.

To enable `emapcc` backend, please install `pybind11` and build it with the following command:

In [ ]:
!bash build_emapcc.sh

`Nextmap` uses `gurobipy` as the default ILP solver. If you need to work on really large designs, please acquire a license from `Gurobi`: https://www.gurobi.com/academia/academic-program-and-licenses/.

In [ ]:
%pip install gurobipy
%pip install scipy
%pip install sqlite3

## How to Customize Your Own Equality Saturation for RTL

### Example 1: Retiming
Take `tests/bad_multiplier.v` as an example. we will show how to use `Nextmap`'s built-in retiming rewrite to save flip-flops.

- Step 0: `Nextmap` takes Yosys JSON format as input.  Run the following Yosys commands to generate the JSON file:

In [ ]:
!yosys -q -p "read_verilog tests/bad_multiplier.v; proc; opt_merge; opt_clean; write_json bad_multiplier.json"

- Step 1: Define your cost model:

In [ ]:
def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    return len(ports[0]) * 1.0  # other types

- Step 2: Run `Nextmap`'s retiming rewrite and extract:

In [ ]:
import emap
import json

TEST_NAME = "bad_multiplier"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()
cnt = 1
while cnt > 0:
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$addu", "$muls", "$mulu"])

    cnt = emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

mod = emap.extracts.ilp.extract_no_techmap(netlist, simple_cost_model, OutputFlag=False)

with open(f"{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

- Step 3: Load the optimized design back to Yosys and compare.

In [ ]:
!yosys -Q -T -p "read_json bad_multiplier.json; stat"

In [ ]:
!yosys -Q -T -p "read_json bad_multiplier_extracted.json; stat"

We save 2 flip-flops by retiming!

### Example 2: Technology Mapping (DSP)

This time let's do something more interesting: technology mapping. We will show how to map a dot product design to DSPs.

- Step 0: Generate the JSON file:

In [ ]:
!yosys -q -p "read_verilog tests/dot_product.v; proc; opt_merge; opt_clean; write_json dot_product.json"

- Step 1: Define your cost model:

In [ ]:
def simple_cost_model(type_: str, *ports) -> float:
    if type_ == "$dff":
        return len(ports[0]) * 1.0
    elif type_ in {"$muls", "$mulu"}:
        return len(ports[0]) * len(ports[1]) * 1.0
    elif type_ in {"$adds", "$addu", "$subs", "$subu"}:
        return min(len(ports[0]) + len(ports[1]), len(ports[2])) * 1.0
    elif type_.startswith("$"): # other types
        return len(ports[0]) * 1.0
    return 0.0  # blackboxes or tech cells

- Step 2: Define your technology mapping rules:

In [ ]:
dsp_rules = {
    "signed_mul_1_stage_26_17_48_bit": {    # rule name
        "requirements": {                   # resource requirements
            "dsp48e2": 1                    # use one DSP48E2
        },
        "hidden_inputs": ["clk"],   # hidden input ports, e.g., clock/reset
        "inputs": ["a", "b"],       # input ports
        "outputs": ["p"],           # output ports
        # and a match pattern in SQL
        "match_sql": """
            SELECT mul1.a, mul1.b, dff1.q
            FROM dffs AS dff1 JOIN aby_cells AS mul1
            ON dff1.d = mul1.y
            WHERE mul1.type = '$muls' AND width_of(mul1.a) <= 26 AND width_of(mul1.b) <= 17 AND width_of(dff1.q) <= 48
        """
    },
    "signed_muladd_1_stage_27_18_48_bit": {
        "requirements": {
            "dsp48e2": 1
        },
        "hidden_inputs": ["clk"],
        "inputs": ["a", "b", "c"],
        "outputs": ["p"],
        "match_sql": """
            SELECT mul1.a, mul1.b, add1.b, dff1.q
            FROM dffs AS dff1 JOIN aby_cells AS mul1 JOIN aby_cells AS add1
            ON dff1.d = add1.y AND mul1.y = add1.a
            WHERE mul1.type = '$muls' AND add1.type = '$adds' AND width_of(mul1.a) <= 27 AND width_of(mul1.b) <= 18 AND width_of(add1.b) <= 48 AND width_of(dff1.q) <= 48
        """
    }
}

You may ask how to generate correct configuration for DSPs to perform the desired operation. Please refer to our prior work `Lakeroad`: https://github.com/gussmith23/lakeroad.

- Step 3: All done. You are ready to run `Nextmap`!

In [ ]:
import emap
import json

TEST_NAME = "dot_product"
SCHEMA_PATH = "emap/schema.sql"
netlist = emap.NetlistDB(SCHEMA_PATH)
with open(f"{TEST_NAME}.json", "r") as f:
    netlist.build_from_json(json.load(f)["modules"]["top"])

netlist.rebuild()
cnt = 1
while cnt > 0:
    comm_matches = emap.rewrites.ematch_comm(netlist, ["$adds", "$addu", "$muls", "$mulu"])
    assoc_to_right_matches = emap.rewrites.ematch_assoc_to_right(netlist, ["$adds", "$addu", "$muls", "$mulu"])
    assoc_to_left_matches = emap.rewrites.ematch_assoc_to_left(netlist, ["$adds", "$addu", "$muls", "$mulu"])
    dff_forward_aby_cell_matches = emap.rewrites.ematch_dff_forward_aby_cell(netlist, ["$adds", "$addu", "$muls", "$mulu"])
    dff_backward_aby_cell_matches = emap.rewrites.ematch_dff_backward_aby_cell(netlist, ["$adds", "$addu", "$muls", "$mulu"])

    cnt = 0
    cnt += emap.rewrites.apply_comm(netlist, comm_matches)
    cnt += emap.rewrites.apply_assoc_to_right(netlist, assoc_to_right_matches)
    cnt += emap.rewrites.apply_assoc_to_left(netlist, assoc_to_left_matches)
    cnt += emap.rewrites.apply_dff_forward_aby_cell(netlist, dff_forward_aby_cell_matches)
    cnt += emap.rewrites.apply_dff_backward_aby_cell(netlist, dff_backward_aby_cell_matches)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
    else:
        print("No rewrites applied, stopping")
    netlist.rebuild()

# techmapping
emap.rewrites.create_tech_tables(netlist, dsp_rules)
emap.rewrites.rewrite_tech(netlist, dsp_rules)

mod = emap.extracts.ilp.extract_techmap_with_limit(netlist, simple_cost_model, dsp_rules, {"dsp48e2": 2}, OutputFlag=False)
with open(f"{TEST_NAME}_extracted.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": mod}}, f, indent=2)

- Step 4: Load the optimized design back to Yosys and compare. To make a fair comparison, this time we call `synth_xilinx` to techmap the design in Yosys.

In [ ]:
!yosys -q -p "read_verilog tests/dot_product.v; synth_xilinx -family xcup; write_json dot_product_xilinx.json"

In [ ]:
!yosys -q -p "read_verilog tests/dsp_blackboxes.v; read_json dot_product_extracted.json; synth_xilinx -family xcup; write_json dot_product_nextmap.json"

In [ ]:
!yosys -Q -T -p "read_json dot_product_xilinx.json; stat"

In [ ]:
!yosys -Q -T -p "read_json dot_product_nextmap.json; stat"

Congrats! You save a 32-bit adder and a 32-bit flip-flop by `Nextmap` and your customized rewrite rules!

### Current Limitations

- `Nextmap` does not support blackbox instances (in progress);
- `Nextmap` does not support multiple clock domains (in progress);
- `Nextmap` does not support acyclic extraction (can be done if necessary);
- `Nextmap` does not convert behavioral Verilog to SQL (in progress).